In [3]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict
import itertools

# load the annotation files
ANNOTATOR1_DIR = "../annotations/annotator1"
ANNOTATOR2_DIR = "../annotations/annotator2"
ANNOTATOR3_DIR = "../annotations/annotator3"

LLM_DIR = "../annotations/llm"
COMBINED_DIR = "../annotations/combined"
os.makedirs(COMBINED_DIR, exist_ok=True)

COLUMN_INDEX = ord('E') - ord('A') # 4
UNCERTAIN_IDX = ord('F') - ord('A') # 5

ERROR_HIERARCHY = ["wrong_modif", "meaning", "realism"]
WRONG_MODIF_IDX = ord('J') - ord('A')   # 9
REALISM_IDX    = ord('K') - ord('A')   # 10
MEANING_IDX    = ord('L') - ord('A')   # 11

In [4]:
# helper functions
# match files by modification type + model
def get_file_map(folder, prefix):
    files = [f for f in os.listdir(folder) if f.endswith(".xlsx") and f.startswith(prefix)]
    return {f[len(prefix):]: os.path.join(folder, f) for f in files}

# make sure labels are the same
def normalize_labels(series):
    mapping = {
        "TRUE": True, "T": True, "1": True, "YES": True,
        "FALSE": False, "F": False, "0": False, "NO": False
    }
    return (
        series.astype(str)
        .str.strip()
        .str.upper()
        .map(mapping)   
        .infer_objects(copy=False)
    )

## check human annotations, make sure nothing is missing

In [5]:
# check to see that everything is filled
ERROR_COLS = ["wrong_modif", "meaning", "realism"]

def validate_annotations_for_all(a1_files, a2_files, a3_files, common_suffixes):
    """Validate all annotator files and collect missing/incomplete cases."""
    issues = []  # collect dictionaries for df later

    for suffix in common_suffixes:
        for annot_id, file_dict in zip(["a1", "a2", "a3"], [a1_files, a2_files, a3_files]):
            fpath = file_dict[suffix]
            df = pd.read_excel(fpath)
            fname = os.path.basename(fpath)

            tf_col = df.columns[COLUMN_INDEX]

            # T/F values
            mask_missing_tf = df[tf_col].isna() | (df[tf_col].astype(str).str.strip() == "")
            for idx in df[mask_missing_tf].index:
                issues.append({
                    "file": fname,
                    "annotator": annot_id,
                    "row": idx + 2,  # +2 for Excel row numbering
                    "issue": "Missing T/F value",
                    "tf_value": None,
                    "wrong_modif": df.loc[idx, "wrong_modif"] if "wrong_modif" in df.columns else None,
                    "meaning": df.loc[idx, "meaning"] if "meaning" in df.columns else None,
                    "realism": df.loc[idx, "realism"] if "realism" in df.columns else None,
                })

            # error values
            for i, row in df.iterrows():
                tf_val = str(row[tf_col]).strip().upper()
                if tf_val in ["F", "FALSE"]:
                    filled_errors = sum(
                        pd.notna(row.get(c)) and str(row.get(c)).strip() != "" for c in ERROR_COLS
                    )
                    if filled_errors == 0:
                        issues.append({
                            "file": fname,
                            "annotator": annot_id,
                            "row": i + 2,
                            "issue": "F row missing error annotation",
                            "tf_value": row[tf_col],
                            "wrong_modif": row.get("wrong_modif", None),
                            "meaning": row.get("meaning", None),
                            "realism": row.get("realism", None),
                        })

            print(f"{fname}: checked {len(df)} rows")

    # convert to DataFrame
    issues_df = pd.DataFrame(issues)
    report_path = os.path.join(COMBINED_DIR, "validation_report.csv")

    if not issues_df.empty:
        issues_df.to_csv(report_path, index=False)
        print(f"\n Validation completed with {len(issues_df)} issues found.")
        print(f"Saved detailed report → {report_path}")
    else:
        print("\n All annotation files passed validation with no issues found!")

    return issues_df

In [6]:
# match files
a1_files = get_file_map(ANNOTATOR1_DIR, "a1_")
a2_files = get_file_map(ANNOTATOR2_DIR, "a2_")
a3_files = get_file_map(ANNOTATOR3_DIR, "a3_")

llm_files = get_file_map(LLM_DIR, "llm_")

common_suffixes = sorted(set(a1_files.keys()) & set(a2_files.keys()) & set(a3_files.keys()) & set(llm_files.keys()))

if not common_suffixes:
    raise ValueError("No matching files found across annotators 1, 2, 3, and LLM.")

print(f"Found {len(common_suffixes)} matching files:")
for suffix in common_suffixes:
    print(f"  - {suffix}")

Found 10 matching files:
  - AAE_chatgpt.xlsx
  - AAE_deepseek.xlsx
  - change_voice_chatgpt.xlsx
  - change_voice_deepseek.xlsx
  - formal_chatgpt.xlsx
  - formal_deepseek.xlsx
  - prepositions_chatgpt.xlsx
  - prepositions_deepseek.xlsx
  - synonym_substitution_chatgpt.xlsx
  - synonym_substitution_deepseek.xlsx


In [13]:
# validate
issues_df = validate_annotations_for_all(a1_files, a2_files, a3_files, common_suffixes)

if not issues_df.empty:
    display(issues_df.head())

a1_AAE_chatgpt.xlsx: checked 133 rows
a2_AAE_chatgpt.xlsx: checked 133 rows
a3_AAE_chatgpt.xlsx: checked 133 rows
a1_AAE_deepseek.xlsx: checked 538 rows
a2_AAE_deepseek.xlsx: checked 538 rows
a3_AAE_deepseek.xlsx: checked 538 rows
a1_change_voice_chatgpt.xlsx: checked 355 rows
a2_change_voice_chatgpt.xlsx: checked 355 rows
a3_change_voice_chatgpt.xlsx: checked 355 rows
a1_change_voice_deepseek.xlsx: checked 597 rows
a2_change_voice_deepseek.xlsx: checked 597 rows
a3_change_voice_deepseek.xlsx: checked 597 rows
a1_formal_chatgpt.xlsx: checked 544 rows
a2_formal_chatgpt.xlsx: checked 544 rows
a3_formal_chatgpt.xlsx: checked 544 rows
a1_formal_deepseek.xlsx: checked 561 rows
a2_formal_deepseek.xlsx: checked 561 rows
a3_formal_deepseek.xlsx: checked 561 rows
a1_prepositions_chatgpt.xlsx: checked 147 rows
a2_prepositions_chatgpt.xlsx: checked 147 rows
a3_prepositions_chatgpt.xlsx: checked 147 rows
a1_prepositions_deepseek.xlsx: checked 377 rows
a2_prepositions_deepseek.xlsx: checked 377 row

## compute new ground truth file + prepare for llm judge

In [7]:
def is_marked(cell):
    """Return True if an error cell should be considered marked."""
    if pd.isna(cell):
        return False
    s = str(cell).strip().lower()
    return s not in ("", "0", "false", "f", "no", "nan")

for suffix in common_suffixes:
    df1 = pd.read_excel(a1_files[suffix])
    df2 = pd.read_excel(a2_files[suffix])
    df3 = pd.read_excel(a3_files[suffix])
    df_llm = pd.read_excel(llm_files[suffix])

    col_name = df1.columns[COLUMN_INDEX]  # T/F column
    col1 = normalize_labels(df1.iloc[:, COLUMN_INDEX])
    col2 = normalize_labels(df2.iloc[:, COLUMN_INDEX])
    col3 = normalize_labels(df3.iloc[:, COLUMN_INDEX])
    col_llm = normalize_labels(df_llm.iloc[:, COLUMN_INDEX])

    # creating gt files
    consensus_TF = []
    for v1, v2, v3 in zip(col1, col2, col3):
        votes = [v for v in (v1, v2, v3) if not pd.isna(v)]
        if not votes:
            consensus_TF.append("")
            continue
        t_count = sum(v is True for v in votes)
        f_count = len(votes) - t_count
        consensus_TF.append("T" if t_count > f_count else "F")

    # consensus error only for false rows
    consensus_error = []
    for i, tf in enumerate(consensus_TF):
        if tf == "F":
            if any(is_marked(df.iloc[i, WRONG_MODIF_IDX]) for df in (df1, df2, df3)):
                consensus_error.append("wrong_modif")
            elif any(is_marked(df.iloc[i, MEANING_IDX]) for df in (df1, df2, df3)):
                consensus_error.append("meaning")
            elif any(is_marked(df.iloc[i, REALISM_IDX]) for df in (df1, df2, df3)):
                consensus_error.append("realism")
            else:
                consensus_error.append("")
        else:
            consensus_error.append("")

    # compute uncertain flag
    consensus_uncertain = []
    for i in range(len(consensus_TF)):
        if any(is_marked(df.iloc[i, UNCERTAIN_IDX]) for df in (df1, df2, df3)):
            consensus_uncertain.append("1")
        else:
            consensus_uncertain.append("")

    # build combined dataframe
    combined_df = df1.copy()
    combined_df.rename(columns={col_name: "a1_TF"}, inplace=True)
    combined_df["a2_TF"] = pd.Series(col2).map({True: "T", False: "F"}).fillna("")
    combined_df["a3_TF"] = pd.Series(col3).map({True: "T", False: "F"}).fillna("")
    combined_df["human_TF"] = consensus_TF
    combined_df["human_error"] = consensus_error
    combined_df["human_uncertain"] = consensus_uncertain
    
    # placeholders for LLM judgment (need to consolidate after)
    combined_df["llm_TF"] = col_llm  
    combined_df["llm_error"] = ""
    
    for col in ["wrong_modif", "realism", "meaning"]:
        if col in combined_df.columns:
            combined_df.drop(columns=col, inplace=True)

    # reorder columns
    def reorder_columns(df):
        cols = df.columns.tolist()
        
        # Default to index 4 if not found (E column)
        a1_idx = cols.index("a1_TF") if "a1_TF" in cols else 4
        
        desired_block = [
            "a1_TF",
            "a2_TF",
            "a3_TF",
            "human_uncertain",
            "human_TF",
            "human_error",
            "llm_TF",
            "llm_error",
        ]
        
        # Keep only existing ones (some may not exist yet)
        existing_block = [c for c in desired_block if c in cols]
        
        # Remove them from the rest to avoid duplicates
        remaining_cols = [c for c in cols if c not in existing_block]
        
        # Split remaining columns before and after a1_TF
        if "a1_TF" in cols:
            prefix = remaining_cols[: cols.index("a1_TF")]
            suffix = remaining_cols[cols.index("a1_TF") + 1 :]
        else:
            prefix, suffix = [], remaining_cols
        
        new_order = prefix + existing_block + suffix
        return df[new_order]

    combined_df = reorder_columns(combined_df)
    
    # save output
    combined_path = os.path.join(COMBINED_DIR, f"combined_{suffix}")
    combined_df.to_excel(combined_path, index=False)

    print(f"Saved combined file → {combined_path}")

Saved combined file → ../annotations/combined/combined_AAE_chatgpt.xlsx
Saved combined file → ../annotations/combined/combined_AAE_deepseek.xlsx
Saved combined file → ../annotations/combined/combined_change_voice_chatgpt.xlsx
Saved combined file → ../annotations/combined/combined_change_voice_deepseek.xlsx
Saved combined file → ../annotations/combined/combined_formal_chatgpt.xlsx
Saved combined file → ../annotations/combined/combined_formal_deepseek.xlsx
Saved combined file → ../annotations/combined/combined_prepositions_chatgpt.xlsx
Saved combined file → ../annotations/combined/combined_prepositions_deepseek.xlsx
Saved combined file → ../annotations/combined/combined_synonym_substitution_chatgpt.xlsx
Saved combined file → ../annotations/combined/combined_synonym_substitution_deepseek.xlsx


## compute the agreement scores

In [8]:
import itertools

def tf_to_val(s):
    if s in ("T", True, "1", 1):
        return 1
    elif s in ("F", False, "0", 0):
        return 0
    else:
        return np.nan

def per_question_hh_agreement(row_vals):
    """
    Human–Human agreement for one question.
    row_vals: array of human votes (e.g., [a1, a2, a3]) with NaNs allowed.
    Returns NaN if fewer than 2 humans labeled the question.
    """
    vals = row_vals[~np.isnan(row_vals)]
    if len(vals) < 2:
        return np.nan
    pairs = list(itertools.combinations(vals, 2))
    matches = [1.0 if x == y else 0.0 for x, y in pairs]
    return np.mean(matches)

def per_question_majority(vals):
    """
    Majority label for one question among humans.
    Returns (majority_label, is_tie, total_valid)
    majority_label in {0,1} when not tie; None when tie or no labels.
    """
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        return None, False, 0
    c1 = np.sum(vals == 1)
    c0 = np.sum(vals == 0)
    if c1 > c0:
        return 1, False, c1 + c0
    elif c0 > c1:
        return 0, False, c1 + c0
    else:
        return None, True, c1 + c0  # tie

def per_question_llm_vs_human(vals, llm):
    """
    LLM–Human agreement for one question = fraction of available humans that match the LLM.
    Returns NaN if LLM is NaN or no human labels.
    """
    if np.isnan(llm):
        return np.nan
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        return np.nan
    return np.mean(vals == llm)

def per_question_llm_vs_majority(vals, llm):
    """
    LLM–Majority agreement for one question.
    - strict majority: 1 if LLM == majority else 0
    - tie: 0.5
    Returns NaN if LLM is NaN or no human labels.
    """
    if np.isnan(llm):
        return np.nan
    majority, is_tie, total = per_question_majority(vals)
    if total == 0:
        return np.nan
    if is_tie:
        return 0.5
    return 1.0 if llm == majority else 0.0

def per_question_human_vs_majority(vals):
    """
    Human–Majority agreement upper bound for one question:
    - strict majority: max(count1, count0)/total
    - tie: 0.5
    Returns NaN if no human labels.
    """
    majority, is_tie, total = per_question_majority(vals)
    if total == 0:
        return np.nan
    if is_tie:
        return 0.5
    # compute p(random human == majority) = majority_count / total
    vals = vals[~np.isnan(vals)]
    maj_count = np.sum(vals == majority)
    return maj_count / total

def compute_agreements_for_df(df):
    # Build human matrix (N x H); here H=3 (a1,a2,a3)
    H = np.vstack([
        df["a1_TF"].map(tf_to_val).to_numpy(),
        df["a2_TF"].map(tf_to_val).to_numpy(),
        df["a3_TF"].map(tf_to_val).to_numpy(),
    ]).T  # shape (n_questions, 3)

    llm = df["llm_TF"].map(tf_to_val).to_numpy() if "llm_TF" in df.columns else np.full(len(df), np.nan)

    # Per-question metrics
    hh = np.array([per_question_hh_agreement(H[i, :]) for i in range(H.shape[0])])
    hm = np.array([per_question_human_vs_majority(H[i, :]) for i in range(H.shape[0])])
    lh = np.array([per_question_llm_vs_human(H[i, :], llm[i]) for i in range(H.shape[0])])
    lm = np.array([per_question_llm_vs_majority(H[i, :], llm[i]) for i in range(H.shape[0])])

    # Global averages (question-uniform; ignore NaNs)
    return {
        "human_human_agree": np.nanmean(hh),
        "human_majority_agree": np.nanmean(hm),          # upper bound for LLM–Human
        "llm_vs_human": np.nanmean(lh),
        "llm_vs_human_majority": np.nanmean(lm)
    }

In [10]:
summary = []
files = [f for f in os.listdir(COMBINED_DIR) if f.endswith(".xlsx")]

for fname in files:
    fpath = os.path.join(COMBINED_DIR, fname)
    df = pd.read_excel(fpath)

    base = os.path.splitext(fname)[0].replace("combined_", "").replace("annotated_", "")
    parts = base.split("_")
    modif = "_".join(parts[:-1]) if len(parts) >= 2 else base
    model = parts[-1] if len(parts) >= 2 else "unknown"

    metrics = compute_agreements_for_df(df)
    summary.append({
        "modification": modif,
        "model": model,
        **metrics
    })

summary_df = pd.DataFrame(summary).sort_values(["modification", "model"]).reset_index(drop=True)
summary_df

,modification,model,human_human_agree,human_majority_agree,llm_vs_human,llm_vs_human_majority
0,AAE,chatgpt,0.428571,0.714286,0.518797,0.473684
1,AAE,deepseek,0.413879,0.706939,0.545229,0.578067
2,change_voice,chatgpt,0.716432,0.858216,0.657277,0.684507
3,change_voice,deepseek,0.595757,0.797878,0.606365,0.639866
4,formal,chatgpt,0.639706,0.819853,0.751225,0.867647
5,formal,deepseek,0.387998,0.693999,0.606061,0.850267
6,prepositions,chatgpt,0.555556,0.777778,0.337868,0.183673
7,prepositions,deepseek,0.577365,0.788683,0.461538,0.358090
8,synonym_substitution,chatgpt,0.371111,0.685556,0.546111,0.641667
9,synonym_substitution,deepseek,0.396667,0.698333,0.534444,0.571667


In [21]:
summary = []

for fname in files:
    fpath = os.path.join(COMBINED_DIR, fname)
    df = pd.read_excel(fpath)

    # Parse filename for identifiers
    base = os.path.splitext(fname)[0].replace("combined_", "")
    parts = base.split("_")
    modif = "_".join(parts[:-1]) if len(parts) >= 2 else base
    model = parts[-1] if len(parts) >= 2 else "unknown"

    # 1) Metrics on ALL rows
    metrics_all = compute_agreements_for_df(df)

    #print(df)

    # 2) Filter out only rows explicitly marked as 1 in UNCERTAIN_IDX
    if "human_uncertain" in df.columns:
        # drop rows where UNCERTAIN_IDX == 1
        mask_uncertain = df["human_uncertain"] == 1
        df_filtered = df.loc[~mask_uncertain].copy()
    else:
        df_filtered = df

    metrics_filtered = compute_agreements_for_df(df_filtered)

    summary.append({
        "modification": modif,
        "model": model,

        # all rows
        #"human_human_agree_all": metrics_all["human_human_agree"],
        #"human_majority_agree_all": metrics_all["human_majority_agree"],
        #"llm_vs_human_all": metrics_all["llm_vs_human"],
        #"llm_vs_human_majority_all": metrics_all["llm_vs_human_majority"],

        # filtered (drop rows where UNCERTAIN_IDX == 1)
        "human_human_agree_filtered": metrics_filtered["human_human_agree"],
        "human_majority_agree_filtered": metrics_filtered["human_majority_agree"],
        "llm_vs_human_filtered": metrics_filtered["llm_vs_human"],
        "llm_vs_human_majority_filtered": metrics_filtered["llm_vs_human_majority"],

        "n_rows_all": len(df),
        "n_rows_filtered": len(df_filtered),
        #"n_rows_dropped_uncertain": mask_uncertain.sum(),
    })

summary_df = (
    pd.DataFrame(summary)
      .sort_values(["modification", "model"])
      .reset_index(drop=True)
)

summary_df

,modification,model,human_human_agree_filtered,human_majority_agree_filtered,llm_vs_human_filtered,llm_vs_human_majority_filtered,n_rows_all,n_rows_filtered
0,AAE,chatgpt,0.406061,0.703030,0.518182,0.463636,133,110
1,AAE,deepseek,0.414996,0.707498,0.556050,0.594655,538,449
2,change_voice,chatgpt,0.758270,0.879135,0.717557,0.744275,355,262
3,change_voice,deepseek,0.615504,0.807752,0.638760,0.688372,597,430
4,formal,chatgpt,0.646192,0.823096,0.787060,0.931204,544,407
5,formal,deepseek,0.385714,0.692857,0.625397,0.892857,561,420
6,prepositions,chatgpt,0.551724,0.775862,0.295977,0.112069,147,116
7,prepositions,deepseek,0.581493,0.790747,0.453207,0.353312,377,317
8,synonym_substitution,chatgpt,0.359760,0.679880,0.549550,0.648649,600,555
9,synonym_substitution,deepseek,0.390018,0.695009,0.536044,0.574861,600,541
